# Real-Time Emotion Detection — ResNet50 + CBAM

Understanding, building, and verifying a from-scratch **CBAM** (Convolutional Block Attention Module) classifier on **FER+** (8 emotions), then wrapping it in a real-time Streamlit app.

**Pipeline:** Google Colab (T4 GPU) for data + training + evaluation; Streamlit for image upload + webcam inference; deployed on Streamlit Community Cloud.

Checkpoints saved to Google Drive: `/content/drive/MyDrive/emotion_detection/`.

## M1 — Convolutional Block Attention Module, implemented from scratch

Reference: Woo et al., *CBAM: Convolutional Block Attention Module*, ECCV 2018.

A plain CNN treats **every channel and pixel equally**. CBAM says: some channels (filters) matter more than others for this image, and some spatial locations matter more. It learns to rescale both — first **which channels** (`C`), then **where** in space (`H,W`).

**Channel attention** `[B,C,H,W] -> [B,C,1,1]` (per-channel importance):
- Squeeze each channel to a single number, twice: `AdaptiveAvgPool2d(1)` and `AdaptiveMaxPool2d(1)`. Average captures the *overall* response, max captures the *most salient* response — together they cover more than either alone.
- Pass both through a **shared MLP** (bottleneck with reduction ratio `r`: `C -> C/r -> C`), add the two outputs, `sigmoid`.
- Scale every spatial position of each channel by its sigmoid weight.

**Spatial attention** `[B,C,H,W] -> [B,1,H,W]` (per-location importance):
- Pool across the channel axis: channel-wise `mean` and `max` -> 2 maps `[B,1,H,W]` each, concatenate -> `[B,2,H,W]`.
- `7x7` convolution -> `[B,1,H,W]`, `sigmoid`.
- Scale every channel at each location by that spatial weight.

Order matters (from the paper): channel attention first, then spatial. Both are cheap (`~400k` params on a ResNet50 = ~1.7% overhead).

In this milestone we implement the modules ourselves, verify output shapes, confirm gradients flow, and count parameters.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        hidden = max(in_channels // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, in_channels),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        avg = self.avg_pool(x).view(b, c)
        max_ = self.max_pool(x).view(b, c)
        attn = self.mlp(avg) + self.mlp(max_)
        attn = self.sigmoid(attn).view(b, c, 1, 1)
        return x * attn

In [ ]:
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        max_ = torch.max(x, dim=1, keepdim=True)[0]
        attn = torch.cat([avg, max_], dim=1)
        attn = self.sigmoid(self.conv(attn))
        return x * attn

In [ ]:
class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel = ChannelAttention(in_channels, reduction)
        self.spatial = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.channel(x)
        x = self.spatial(x)
        return x

### Verify shapes — the module must never change `[B,C,H,W]`

We also extract the *intermediate* attention masks to see exactly what shape the channel vs spatial parts produce.

In [ ]:
def channel_mask(mod, x):
    b, c, _, _ = x.size()
    avg = mod.avg_pool(x).view(b, c)
    max_ = mod.max_pool(x).view(b, c)
    out = mod.mlp(avg) + mod.mlp(max_)
    return mod.sigmoid(out).view(b, c, 1, 1)


def spatial_mask(mod, x):
    avg = torch.mean(x, dim=1, keepdim=True)
    max_ = torch.max(x, dim=1, keepdim=True)[0]
    xcat = torch.cat([avg, max_], dim=1)
    return mod.sigmoid(mod.conv(xcat))


x = torch.randn(4, 64, 56, 56)
c = ChannelAttention(64)
s = SpatialAttention()

print("input           :", tuple(x.shape))
print("channel mask    :", tuple(channel_mask(c, x).shape))
print("spatial mask    :", tuple(spatial_mask(s, x).shape))
print("channel output  :", tuple(c(x).shape))
print("spatial output  :", tuple(s(x).shape))

cb = CBAM(64)
print("CBAM output     :", tuple(cb(x).shape))
assert tuple(cb(x).shape) == (4, 64, 56, 56), "CBAM must preserve shape"
print("shape contract OK")

In [ ]:
# gradients must flow into the attention weights
cb = CBAM(128)
x = torch.randn(2, 128, 28, 28, requires_grad=True)
y = cb(x).mean()
y.backward()

params = [p for p in cb.parameters() if p.requires_grad]
missing = [n for n, p in cb.named_parameters() if p.grad is None]
print("trainable params:", len(params))
print("params with no gradient:", missing if missing else "none")
print("mean |grad| channel mlp w0: %.2e" % cb.channel.mlp[0].weight.grad.abs().mean())
print("mean |grad| spatial conv w: %.2e" % cb.spatial.conv.weight.grad.abs().mean())

### Visualize the masks

Feed a small random 'image' through CBAM and look at (1) the 1D channel weights and (2) the 2D spatial importance map. In a real image, the spatial map concentrates on the informative region (here: the face); on noise it stays flat.

In [ ]:
x = torch.randn(1, 8, 16, 16)
ch = ChannelAttention(8)
sp = SpatialAttention()

cm = channel_mask(ch, x).squeeze().detach().numpy()
sm = spatial_mask(sp, x).squeeze().detach().numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].imshow(x[0, 0].detach().numpy(), cmap="gray")
axes[0].set_title("input channel 0")
axes[1].bar(range(len(cm)), cm)
axes[1].set_title("channel weights")
axes[1].set_xlabel("channel")
im = axes[2].imshow(sm, cmap="inferno")
axes[2].set_title("spatial importance")
plt.colorbar(im, ax=axes[2])
plt.tight_layout()
plt.show()

In [ ]:
class ResNet50_CBAM(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.cbam3 = CBAM(1024)
        self.cbam4 = CBAM(2048)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.cbam3(x)
        x = self.layer4(x)
        x = self.cbam4(x)
        x = self.avgpool(x)
        return self.classifier(x)

In [ ]:
model = ResNet50_CBAM(num_classes=8).to(device)

total = sum(p.numel() for p in model.parameters())
cbam_params = sum(p.numel() for p in model.cbam3.parameters()) + sum(p.numel() for p in model.cbam4.parameters())
base = total - cbam_params
imagenet_fc = models.resnet50().fc

print(f"total params           : {total/1e6:7.3f} M")
print(f"our head w/o CBAM      : {base/1e6:7.3f} M")
print(f"CBAM overhead          : {cbam_params/1e3:7.2f} k  ({100*cbam_params/base:.2f}%)")
print(f"ImageNet fc replaced   : {sum(p.numel() for p in imagenet_fc.parameters())/1e6:.2f} M -> Linear(2048,512)+Linear(512,8)")


### M1 done — checklist to confirm
- Channel & spatial attention implemented from scratch, CBAM preserves `[B,C,H,W]`.
- Attention masks have the right shapes (`[B,C,1,1]` channel, `[B,1,H,W]` spatial) and gradients flow into both.
- `ResNet50_CBAM(8)` builds and forward-passes on a real input; total params ~25.98 M (CBAM = ~1.7% overhead).

Next: **M2** — build the FER+ 8-class dataset and look at its (heavily imbalanced) class distribution.

## M2 \u2014 Build the FER+ 8-class dataset

FER+ (Barsoum et al., 2016) re-labels the classic FER2013 images: each of the 35,887 grayscale 48x48 faces was re-annotated by **10 crowd workers** into 8 emotion classes plus `unknown` and `NF` (not a face). Microsoft publishes only the vote-count CSV (`fer2013new.csv`), which is **row-aligned with the original fer2013.csv**.

Our recipe:
1. Download the raw original file `fer2013.csv.zip` (48x48 grayscale pixel strings + `Usage`) and unzip it.
2. Fetch `fer2013new.csv` \u2014 row `i` in `fer2013new.csv` maps to row `i` in `fer2013.csv`.
3. Majority vote: assign the emotion that collected a strict majority (>50%) of the votes; otherwise mark as `unknown`/`NF` and drop.
4. Re-split using the `Usage` column (Training / PublicTest / PrivateTest).

Label order we adopt (matches the FER+ column order): `[neutral, happiness, surprise, sadness, anger, disgust, fear, contempt]`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
save_dir = '/content/drive/MyDrive/emotion_detection'
os.makedirs(save_dir, exist_ok=True)
print('drive dir:', save_dir)

In [ ]:
import os, urllib.request, zipfile

zip_path = 'fer2013.csv.zip'
if not os.path.exists(zip_path):
    urllib.request.urlretrieve(
        'https://huggingface.co/datasets/chitradrishti/fer2013/resolve/main/fer2013.csv.zip',
        zip_path)
with zipfile.ZipFile(zip_path) as z:
    print('zip contents:', z.namelist())
    z.extractall('.')

### Local check \u2014 raw CSV, all 35,887 rows in original order

`load_dataset` does not support this repo (raw CSV zip, not parquet), so we download `fer2013.csv.zip` directly. It holds the pristine `fer2013.csv` (columns `emotion, pixels, Usage`) in the original Kaggle row order: the Training block first, then PublicTest, then PrivateTest.

In [ ]:
import pandas as pd, glob

fer2013_path = 'fer2013.csv'
if not os.path.exists(fer2013_path):
    fer2013_path = sorted(glob.glob('**/fer2013.csv', recursive=True))[0]

df = pd.read_csv(fer2013_path)
print('shape:', df.shape)
print('columns:', df.columns.tolist())
print('Usage counts:')
print(df['Usage'].value_counts())
print('first usage:', df['Usage'].head(3).tolist(), '| last usage:', df['Usage'].tail(3).tolist(), )
assert len(df) == 35887, 'expected 35,887 rows'

In [ ]:
import urllib.request

url = 'https://raw.githubusercontent.com/microsoft/FERPlus/master/fer2013new.csv'
urllib.request.urlretrieve(url, 'fer2013new.csv')
labels = pd.read_csv('fer2013new.csv')
print(labels.shape)
print(labels.columns.tolist())

### Majority-vote labels

The vote columns are: `neutral, happiness, surprise, sadness, anger, disgust, fear, contempt, unknown, NF`.

Per the FER+ paper, we take the strict majority (>50% of the votes). If no emotion reaches a majority, the image was ambiguous -> mark `unknown`. We drop `unknown` and `NF` rows (a few %). `NF` = the workers judged it not a face.


In [ ]:
FER_PLUS_COLS = ['neutral', 'happiness', 'surprise', 'sadness', 'anger', 'disgust', 'fear', 'contempt', 'unknown', 'NF']
CLS_NAMES = FER_PLUS_COLS[:8]

votes = labels[FER_PLUS_COLS].values.astype(int).copy()
assert len(labels) == len(df), 'fer2013new.csv must be row-aligned with fer2013.csv'
totals = votes.sum(axis=1)
majority_idx = votes.argmax(axis=1)
is_majority = votes[np.arange(len(votes)), majority_idx] > 0.5 * totals
is_face = majority_idx < 8  # emotions are columns 0..7; unknown=8, NF=9

df['cls'] = np.where(is_majority & is_face, majority_idx, -1)
df['valid'] = (df['cls'] >= 0)
print('valid (labeled face with majority):', int(df['valid'].sum()), '/', len(df), f'({df["valid"].mean():.1%})')
print('dropped as ambiguous/NF/unknown    :', int((~df['valid']).sum()))

In [ ]:
def split_rows(usage):
    return df[df['Usage'] == usage]

def class_counts(sub):
    return sub[sub['valid']].groupby('cls').size()

train_rows = split_rows('Training')
val_rows = split_rows('PublicTest')
test_rows = split_rows('PrivateTest')

for name, sub in [('train', train_rows), ('val', val_rows), ('test', test_rows)]:
    cc = class_counts(sub)
    print(f"{name:6s} total={len(sub):5d} valid={int(sub['valid'].sum()):5d} ")
    print('   ' + ', '.join(f'{CLS_NAMES[i]}:{cc.get(i,0)}' for i in range(8)))

### Class imbalance — the central data problem of FER+

Expected: `neutral`/`happiness` dominate; `contempt`, `disgust`, `fear` are rare (contempt often ~1-2% of the set).

We will care about:
- overall **accuracy is misleading** when classes are imbalanced (a neutral/happy-only model scores high), so rely on per-class `F1` / macro-F1 and a confusion matrix;
- training decisions in M3/M4 (class-weighted loss, and possibly oversampling the rare classes);
- detecting *shortcut learning*: e.g. the model mapping everything to neutral/happiness.

Look at the bar chart and the per-class counts below.

In [ ]:
import matplotlib.pyplot as plt

cc = class_counts(train_rows)
plt.figure(figsize=(8, 3.5))
plt.bar(range(8), [cc.get(i, 0) for i in range(8)], color='teal')
plt.xticks(range(8), CLS_NAMES, rotation=30)
plt.title('FER+ class counts (train)')
plt.ylabel('count')
plt.tight_layout()
plt.show()

In [ ]:
df.to_pickle(os.path.join(save_dir, 'ferplus_rows.pkl'))
print('saved ferplus_rows.pkl to', save_dir)

### M2 done — checklist
- Loaded FER2013 images, restored original row order (`Unnamed: 0` matches 0..35886).
- Joined the Microsoft FER+ vote labels row-aligned.
- Majority-vote labels -> 8 classes; ambiguous/`NF` dropped.
- Repartitioned by `Usage` into train / val / test; distribution printed + plotted.
- **Assess the imbalance before deciding M3/M4 training loss.**

Next: **M3** — build loaders (48x48 gray -> 3-channel -> Resize 224) and fine-tune the baseline ResNet50 (no CBAM).

### Imbalance strategy

FER+ is extreme: disgust=46, contempt=78 vs neutral=7,495. We counter it with **class-weighted loss only**:

`w_c = min(N / (K*n_c), 8)` renormalized to mean 1 — rare classes cost more, but greedily capped so the head can't collapse onto the rare classes. We deliberately do **not** add a `WeightedRandomSampler`: with weights AND oversampling the rare classes get double-driven and validation collapses. We also keep the loss-weighted macro-F1 as our model-selection metric.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm
from PIL import Image

torch.manual_seed(42)
np.random.seed(42)

batch_size = 64
num_workers = 2
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])

In [ ]:
class FERPlusDataset(torch.utils.data.Dataset):
    def __init__(self, df_rows, transform):
        self.rows = df_rows[df_rows['valid']].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows.iloc[idx]
        pix = np.array([int(v) for v in r['pixels'].split()], dtype=np.uint8).reshape(48, 48)
        img = Image.fromarray(pix).convert('RGB')
        return self.transform(img), int(r['cls'])


train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

df = pd.read_pickle(os.path.join(save_dir, 'ferplus_rows.pkl'))
train_rows = df[df['Usage'] == 'Training']
val_rows = df[df['Usage'] == 'PublicTest']
test_rows = df[df['Usage'] == 'PrivateTest']

train_ds = FERPlusDataset(train_rows, train_tf)
val_ds = FERPlusDataset(val_rows, eval_tf)
test_ds = FERPlusDataset(test_rows, eval_tf)

print('train / val / test:', len(train_ds), len(val_ds), len(test_ds))

In [ ]:
from torch.utils.data import DataLoader

counts = train_ds.rows.groupby('cls').size().astype(float)
counts = counts.reindex(range(8), fill_value=0.0).values

# FER+ imbalance is extreme (disgust 46 vs neutral 7495). We apply per-class
# loss weights ONLY, capped at 8x, renormalized so the mean is 1. We do NOT
# additionally oversample with a sampler - weights + sampler together over-drive
# the head toward the rare classes and crush validation accuracy.
raw_weights = counts.sum() / (8 * counts)
weights = np.minimum(raw_weights, 8.0)
weights = weights / weights.mean()

criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32).to(device))

print('class weights (mean 1, capped at 8):', np.round(weights, 2))

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                         num_workers=num_workers, pin_memory=True)


### The baseline model: identical to M4 minus the CBAM blocks

Same stem, same `layer1..4`, same classifier `Linear(2048,512)->ReLU->Dropout(0.5)->Linear(512,8)`. The only difference to M4 is the two CBAM modules after `layer3`/`layer4` — that's what makes the ablation clean.

In [ ]:
class ResNet50_Base(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        return self.classifier(x)


baseline = ResNet50_Base(num_classes=8).to(device)
print('baseline params: %.3f M' % (sum(p.numel() for p in baseline.parameters()) / 1e6))
x = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    print('forward:', tuple(baseline(x).shape))

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for images, labels in tqdm(loader, desc='train', leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct += (out.argmax(1) == labels).sum().item()
        n += len(labels)
    return total_loss / n, correct / n


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='eval', leave=False):
            images, labels = images.to(device), labels.to(device)
            out = model(images)
            total_loss += criterion(out, labels).item() * len(labels)
            preds = out.argmax(1).cpu()
            correct += (preds == labels.cpu()).sum().item()
            n += len(labels)
            all_preds.append(preds)
            all_labels.append(labels.cpu())
    y_true = torch.cat(all_labels).numpy()
    y_pred = torch.cat(all_preds).numpy()
    acc = correct / n
    rep = classification_report(y_true, y_pred, labels=list(range(8)), target_names=CLS_NAMES,
                                output_dict=True, zero_division=0)
    macro_f1 = rep['macro avg']['f1-score']
    return total_loss / n, acc, macro_f1, y_true, y_pred


def make_optimizer(model, backbone_lr, head_lr):
    head_ids = set(id(p) for p in model.classifier.parameters())
    base = [p for p in model.parameters() if id(p) not in head_ids and p.requires_grad]
    head = [p for p in model.classifier.parameters()]
    return torch.optim.Adam([
        {'params': base, 'lr': backbone_lr},
        {'params': head, 'lr': head_lr},
    ])

### Training

Stage 1 (head only, 4 epochs) warms the new classifier against the frozen features. Stage 2 (full fine-tune, 8 epochs) adapts the features. We keep the best **val macro-F1** checkpoint for test evaluation and plot both loss curves.

In [ ]:
baseline_path = os.path.join(save_dir, 'emotion_baseline.pt')

def set_requires_grad(module, value):
    for p in module.parameters():
        p.requires_grad = value

# stage 1: head only
set_requires_grad(baseline, False)
set_requires_grad(baseline.classifier, True)
opt = torch.optim.Adam([{"params": baseline.classifier.parameters()}], lr=1e-3)

best_f1 = 0.0
history = {'loss': [], 'val_loss': []}

for epoch in range(4):
    tr_loss, tr_acc = train_one_epoch(baseline, train_loader, criterion, opt, device)
    va_loss, va_acc, va_f1, _, _ = evaluate(baseline, val_loader, criterion, device)
    history['loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    print(f"s1 e{epoch+1}/4  train {tr_loss:.4f}/{tr_acc:.4f}  val {va_loss:.4f}/{va_acc:.4f} f1 {va_f1:.4f}")
    if va_f1 > best_f1:
        best_f1 = va_f1
        torch.save(baseline.state_dict(), baseline_path)
        print('  saved best')


In [ ]:
# stage 2: full fine-tune, differential lr
set_requires_grad(baseline, True)
opt = make_optimizer(baseline, backbone_lr=1e-4, head_lr=1e-3)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=2)

for epoch in range(8):
    tr_loss, tr_acc = train_one_epoch(baseline, train_loader, criterion, opt, device)
    va_loss, va_acc, va_f1, _, _ = evaluate(baseline, val_loader, criterion, device)
    sched.step(va_loss)
    history['loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    print(f"s2 e{epoch+1}/8  train {tr_loss:.4f}/{tr_acc:.4f}  val {va_loss:.4f}/{va_acc:.4f} f1 {va_f1:.4f}")
    if va_f1 > best_f1:
        best_f1 = va_f1
        torch.save(baseline.state_dict(), baseline_path)
        print('  saved best')

plt.figure(figsize=(6, 3))
plt.plot(history['loss'], label='train')
plt.plot(history['val_loss'], label='val')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.title('baseline loss curve')
plt.tight_layout()
plt.show()
print('best val macro-F1:', round(best_f1, 4))


### Baseline test evaluation

Load the best-val checkpoint and evaluate on the held-out test split. Report overall accuracy AND per-class accuracy with a confusion matrix — on the tiny classes (disgust 7, contempt 9 in test) expect noisy per-class numbers.

In [ ]:
baseline.load_state_dict(torch.load(baseline_path, map_location=device))
baseline.eval()
test_loss, test_acc, test_f1, y_true, y_pred = evaluate(baseline, test_loader, criterion, device)

print(f"baseline test  acc {test_acc:.4f}  macro-F1 {test_f1:.4f}")
print('')
print(classification_report(y_true, y_pred, labels=list(range(8)), target_names=CLS_NAMES, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))
plt.figure(figsize=(6.5, 5.5))
plt.imshow(cm, cmap='Blues')
for i in range(8):
    for j in range(8):
        plt.text(j, i, cm[i, j], ha='center', va='center', color='black', fontsize=8)
plt.xticks(range(8), CLS_NAMES, rotation=45)
plt.yticks(range(8), CLS_NAMES)
plt.xlabel('predicted')
plt.ylabel('true')
plt.title('baseline confusion matrix (FER+ test)')
plt.tight_layout()
plt.show()

### Save the M3 snapshot to Drive

The training loop already saved the best-val checkpoint `emotion_baseline.pt` to Drive on each improvement. Here we also archive the test metrics as JSON so M4 can be compared against a persistent record.

In [ ]:
import json as _json

rep = classification_report(y_true, y_pred, labels=list(range(8)), target_names=CLS_NAMES,
                           output_dict=True, zero_division=0)
snapshot = {
    'model': 'baseline (no CBAM)',
    'test_accuracy': test_acc,
    'test_macro_f1': test_f1,
    'per_class': {k: v for k, v in rep.items() if k in CLS_NAMES},
}
snapshot_path = os.path.join(save_dir, 'm3_baseline_results.json')
with open(snapshot_path, 'w') as f:
    _json.dump(snapshot, f, indent=2) # noqa: E501
print('saved snapshot:', snapshot_path)
print('drive files:')
for fn in sorted(os.listdir(save_dir)):
    fp = os.path.join(save_dir, fn)
    if os.path.isfile(fp):
        print('  %-32s %.1f MB' % (fn, os.path.getsize(fp) / 1e6))

### M3 done — record before moving on
- Test accuracy, macro-F1, and per-class numbers for the **baseline**.
- Note which classes dominate the errors (check the confusion matrix rows for anger/disgust/fear/contempt).
- M4 runs the exact same protocol with CBAM inserted; the only difference will be the two attention modules.

## M4 &mdash; ResNet50 + CBAM: the ablation

Same data, same loss (capped class weights, no sampler), same staging, same seeds &mdash; the **only** difference from M3 is the two CBAM modules added after `layer3` (1024 ch) and `layer4` (2048 ch), built from scratch in M1:

- **Channel attention** (avg+max pool &rarr; shared MLP, reduction 16 &rarr; sigmoid) rescales each channel by how informative it is;
- **Spatial attention** (concat of mean &amp; max across channels &rarr; 7&times;7 conv &rarr; sigmoid) highlights *where* in the face the signal lives (mouth for happiness, eyes/brow for surprise).

CBAM adds ~0.66M params (&asymp;2.6% of the base 25.5M) - a small, honest price to test a real hypothesis. Because FER+ expressions differ largely by local cues on aligned faces, we expect CBAM's largest wins (if any) on the subtle classes (surprise/anger/fear). If the win is a wash, that is a valid, publishable ablation finding - we report what happens rather than cherry-pick.


In [ ]:
torch.manual_seed(42)
np.random.seed(42)

cbam = ResNet50_CBAM(num_classes=8).to(device)
n_total = sum(p.numel() for p in cbam.parameters())
n_train = sum(p.numel() for p in cbam.parameters() if p.requires_grad)
print(f'ResNet50_CBAM params: {n_total:,} (trainable {n_train:,})')
assert n_total == sum(p.numel() for p in ResNet50_Base(num_classes=8).parameters()) + 658_822, 'param delta check'


### Stage 1: head only (4 epochs) - same as M3

Freeze the backbone + CBAM, train the classifier. `set_requires_grad` and `make_optimizer` come from the M3 cells already in this kernel.


In [ ]:
cbam_path = os.path.join(save_dir, 'emotion_cbam.pt')

set_requires_grad(cbam, False)
set_requires_grad(cbam.classifier, True)
opt = torch.optim.Adam([{'params': cbam.classifier.parameters()}], lr=1e-3)

best_f1_c = 0.0
hist_c = {'loss': [], 'val_loss': []}

for epoch in range(4):
    tr_loss, tr_acc = train_one_epoch(cbam, train_loader, criterion, opt, device)
    va_loss, va_acc, va_f1, _, _ = evaluate(cbam, val_loader, criterion, device)
    hist_c['loss'].append(tr_loss)
    hist_c['val_loss'].append(va_loss)
    print(f's1 e{epoch+1}/4  train {tr_loss:.4f}/{tr_acc:.4f}  val {va_loss:.4f}/{va_acc:.4f} f1 {va_f1:.4f}')
    if va_f1 > best_f1_c:
        best_f1_c = va_f1
        torch.save(cbam.state_dict(), cbam_path)
        print('  saved best')


### Stage 2: full fine-tune (8 epochs, differential lr) - same as M3


In [ ]:
set_requires_grad(cbam, True)
opt = make_optimizer(cbam, backbone_lr=1e-4, head_lr=1e-3)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=2)

for epoch in range(8):
    tr_loss, tr_acc = train_one_epoch(cbam, train_loader, criterion, opt, device)
    va_loss, va_acc, va_f1, _, _ = evaluate(cbam, val_loader, criterion, device)
    sched.step(va_loss)
    hist_c['loss'].append(tr_loss)
    hist_c['val_loss'].append(va_loss)
    print(f's2 e{epoch+1}/8  train {tr_loss:.4f}/{tr_acc:.4f}  val {va_loss:.4f}/{va_acc:.4f} f1 {va_f1:.4f}')
    if va_f1 > best_f1_c:
        best_f1_c = va_f1
        torch.save(cbam.state_dict(), cbam_path)
        print('  saved best')

plt.figure(figsize=(6, 3))
plt.plot(hist_c['loss'], label='train')
plt.plot(hist_c['val_loss'], label='val')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.title('CBAM loss curve')
plt.tight_layout()
plt.show()
print('CBAM best val macro-F1:', round(best_f1_c, 4))


### Compare on test - baseline vs CBAM

Load the best-val CBAM checkpoint, score the test split, then diff it cell-by-cell against the archived M3 snapshot (`m3_baseline_results.json`). Units: test accuracy and per-class **F1**. Reminder: disgust (7) and contempt (9) test supports are too small to draw conclusions from.


In [ ]:
import json as _json

cbam.load_state_dict(torch.load(cbam_path, map_location=device))
cbam.eval()
test_loss_c, test_acc_c, test_f1_c, y_true_c, y_pred_c = evaluate(cbam, test_loader, criterion, device)

print(f'CBAM test acc {test_acc_c:.4f}  macro-F1 {test_f1_c:.4f}')
print(classification_report(y_true_c, y_pred_c, labels=list(range(8)), target_names=CLS_NAMES, zero_division=0))

cm_c = confusion_matrix(y_true_c, y_pred_c, labels=list(range(8)))
plt.figure(figsize=(6.5, 5.5))
plt.imshow(cm_c, cmap='Blues')
for i in range(8):
    for j in range(8):
        plt.text(j, i, cm_c[i, j], ha='center', va='center', color='black', fontsize=8)
plt.xticks(range(8), CLS_NAMES, rotation=45)
plt.yticks(range(8), CLS_NAMES)
plt.xlabel('predicted')
plt.ylabel('true')
plt.title('CBAM confusion matrix (FER+ test)')
plt.tight_layout()
plt.show()

base = _json.load(open(os.path.join(save_dir, 'm3_baseline_results.json')))
report_c = classification_report(y_true_c, y_pred_c, labels=list(range(8)), target_names=CLS_NAMES,
                                 output_dict=True, zero_division=0)

cmp_df = pd.DataFrame({
    'class': CLS_NAMES,
    'baseline_F1': [base['per_class'][c]['f1-score'] for c in CLS_NAMES],
    'CBAM_F1': [report_c[c]['f1-score'] for c in CLS_NAMES],
    'delta': [report_c[c]['f1-score'] - base['per_class'][c]['f1-score'] for c in CLS_NAMES],
})
print(cmp_df.round(4).to_string(index=False))
print(f'\ntest acc : baseline {base["test_accuracy"]:.4f} -> CBAM {test_acc_c:.4f}  ({(test_acc_c - base["test_accuracy"]):+.4f})')
print(f'test F1  : baseline {base["test_macro_f1"]:.4f} -> CBAM {test_f1_c:.4f}  ({(test_f1_c - base["test_macro_f1"]):+.4f})')
print('CBAM won if both deltas > 0; we report whatever it is.')

snapshot_c = {
    'model': 'resnet50 + CBAM',
    'test_accuracy': test_acc_c,
    'test_macro_f1': test_f1_c,
    'per_class': {k: v for k, v in report_c.items() if k in CLS_NAMES},
}
with open(os.path.join(save_dir, 'm4_cbam_results.json'), 'w') as f:
    _json.dump(snapshot_c, f, indent=2)
print('\nsaved snapshot: m4_cbam_results.json')
print('drive files:')
for fn in sorted(os.listdir(save_dir)):
    fp = os.path.join(save_dir, fn)
    if os.path.isfile(fp):
        print('  %-32s %.1f MB' % (fn, os.path.getsize(fp) / 1e6))


## M4b &mdash; CBAM + face-detect crop (MTCNN, 15% margin)

The deepfake project (M7.5) showed preprocessing is a *dominant* confounder: a tight MTCNN crop removed context and flipped decisions. We replicate that study for FER+. The 48&times;48 images are already aligned face crops, so the MTCNN box + margin should land near the full frame &mdash; expect a small (possibly zero) effect. Face boxes are computed **once**, cached to Drive, then used to crop each sample before the resize to 224. This is also the exact preprocessing the deployed app's "face-detect" mode will use (M5/M6).


In [ ]:
# face boxes via OpenCV Haar cascade (MTCNN's PNet returns no proposals at 48x48;
# cv2 is preinstalled in Colab and detector choice is not the experiment)
import cv2
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

BOXES_PATH = os.path.join(save_dir, 'ferplus_face_boxes.pkl')

def valid_rows(sub):
    return sub[sub['valid']].reset_index(drop=True)

def boxes_for(rows):
    out = []
    pix = rows['pixels'].tolist()
    for v in tqdm(pix, leave=False):
        arr = np.array([int(x) for x in v.split()], dtype=np.uint8).reshape(48, 48)
        faces = face_cascade.detectMultiScale(arr, scaleFactor=1.1, minNeighbors=5, minSize=(20, 20))
        if len(faces) == 0:
            out.append(None)
        else:
            x, y, w, h = faces[0]
            out.append((max(0, x), max(0, y), min(48, x + w), min(48, y + h)))
    return out

if os.path.exists(BOXES_PATH):
    boxes_pkl = pd.read_pickle(BOXES_PATH)
    print('loaded cached face boxes')
else:
    boxes_pkl = {
        'train': boxes_for(valid_rows(train_rows)),
        'val': boxes_for(valid_rows(val_rows)),
        'test': boxes_for(valid_rows(test_rows)),
    }
    pd.to_pickle(boxes_pkl, BOXES_PATH)
    print('computed and cached face boxes ->', BOXES_PATH)

for k in ('train', 'val', 'test'):
    rate = 1 - sum(1 for b in boxes_pkl[k] if b is None) / len(boxes_pkl[k])
    print(f'face detection rate {k:5s}: {rate:.3f}')


In [ ]:
class FERPlusCropDataset(torch.utils.data.Dataset):
    def __init__(self, rows, boxes, transform, margin=0.15):
        self.rows = rows
        self.boxes = boxes
        self.transform = transform
        self.margin = margin

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows.iloc[idx]
        pix = np.array([int(v) for v in r['pixels'].split()], dtype=np.uint8).reshape(48, 48)
        img = Image.fromarray(pix).convert('RGB')
        b = self.boxes[idx]
        if b is not None:
            x1, y1, x2, y2 = b
            if x2 > x1 and y2 > y1:
                m = int(max(x2 - x1, y2 - y1) * self.margin)
                cx1, cy1 = max(0, x1 - m), max(0, y1 - m)
                cx2, cy2 = min(48, x2 + m), min(48, y2 + m)
                if cx2 > cx1 and cy2 > cy1:
                    img = img.crop((cx1, cy1, cx2, cy2))
        return self.transform(img), int(r['cls'])

vr_train = valid_rows(train_rows)
vr_val = valid_rows(val_rows)
vr_test = valid_rows(test_rows)

fd_train_ds = FERPlusCropDataset(vr_train, boxes_pkl['train'], train_tf)
fd_val_ds = FERPlusCropDataset(vr_val, boxes_pkl['val'], eval_tf)
fd_test_ds = FERPlusCropDataset(vr_test, boxes_pkl['test'], eval_tf)

fd_train_loader = DataLoader(fd_train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
fd_val_loader = DataLoader(fd_val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
fd_test_loader = DataLoader(fd_test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

torch.manual_seed(42)
np.random.seed(42)
cbam_fd = ResNet50_CBAM(num_classes=8).to(device)
print('cbam_fd params:', sum(p.numel() for p in cbam_fd.parameters()))


### Stage 1 (head only) - M4b

In [ ]:
emotion_cbam_fd = os.path.join(save_dir, 'emotion_cbam_fd.pt')

set_requires_grad(cbam_fd, False)
set_requires_grad(cbam_fd.classifier, True)
opt = torch.optim.Adam([{'params': cbam_fd.classifier.parameters()}], lr=1e-3)

best_f1_fd = 0.0
hist_fd = {'loss': [], 'val_loss': []}

for epoch in range(4):
    tr_loss, tr_acc = train_one_epoch(cbam_fd, fd_train_loader, criterion, opt, device)
    va_loss, va_acc, va_f1, _, _ = evaluate(cbam_fd, fd_val_loader, criterion, device)
    hist_fd['loss'].append(tr_loss)
    hist_fd['val_loss'].append(va_loss)
    print(f's1 e{epoch+1}/4  train {tr_loss:.4f}/{tr_acc:.4f}  val {va_loss:.4f}/{va_acc:.4f} f1 {va_f1:.4f}   [fd]')
    if va_f1 > best_f1_fd:
        best_f1_fd = va_f1
        torch.save(cbam_fd.state_dict(), emotion_cbam_fd)
        print('  saved best')


### Stage 2 (full fine-tune) - M4b

In [ ]:
set_requires_grad(cbam_fd, True)
opt = make_optimizer(cbam_fd, backbone_lr=1e-4, head_lr=1e-3)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=2)

for epoch in range(8):
    tr_loss, tr_acc = train_one_epoch(cbam_fd, fd_train_loader, criterion, opt, device)
    va_loss, va_acc, va_f1, _, _ = evaluate(cbam_fd, fd_val_loader, criterion, device)
    sched.step(va_loss)
    hist_fd['loss'].append(tr_loss)
    hist_fd['val_loss'].append(va_loss)
    print(f's2 e{epoch+1}/8  train {tr_loss:.4f}/{tr_acc:.4f}  val {va_loss:.4f}/{va_acc:.4f} f1 {va_f1:.4f}   [fd]')
    if va_f1 > best_f1_fd:
        best_f1_fd = va_f1
        torch.save(cbam_fd.state_dict(), emotion_cbam_fd)
        print('  saved best')

plt.figure(figsize=(6, 3))
plt.plot(hist_fd['loss'], label='train')
plt.plot(hist_fd['val_loss'], label='val')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.title('fd loss curve')
plt.tight_layout()
plt.show()
print('fd best val macro-F1:', round(best_f1_fd, 4))


### Test eval + snapshot - M4b

In [ ]:
import json as _json

cbam_fd.load_state_dict(torch.load(emotion_cbam_fd, map_location=device))
cbam_fd.eval()
test_loss_fd, test_acc_fd, test_f1_fd, y_true_fd, y_pred_fd = evaluate(cbam_fd, fd_test_loader, criterion, device)

print(f'CBAM face-crop test acc {test_acc_fd:.4f}  macro-F1 {test_f1_fd:.4f}')
report_fd = classification_report(y_true_fd, y_pred_fd, labels=list(range(8)), target_names=CLS_NAMES, output_dict=True, zero_division=0)
print(classification_report(y_true_fd, y_pred_fd, labels=list(range(8)), target_names=CLS_NAMES, zero_division=0))

snapshot_fd = {
    'model': 'CBAM face-crop',
    'test_accuracy': test_acc_fd,
    'test_macro_f1': test_f1_fd,
    'per_class': {k: v for k, v in report_fd.items() if k in CLS_NAMES},
}
with open(os.path.join(save_dir, 'm4b_fd_results.json'), 'w') as f:
    _json.dump(snapshot_fd, f, indent=2)
print('saved m4b_fd_results.json')


## M4c &mdash; CBAM + aspect-preserving crop

Deepfake's third mode was "aspect crop": keep the aspect ratio (no squashing), scale the short side, center-crop to 224. Our inputs are square 48&times;48, so an true aspect-crop is degenerate &mdash; instead we apply the same *idea* as a mild center zoom: resize to 256 then center-crop 224 (a ~14% zoom over plain 224). If preprocessing is truly not a confounder for FER+, this should match M4a within noise; we measure it to *demonstrate* that claim, mirroring the deepfake stack.


In [ ]:
ac_train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
ac_eval_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

ac_train_ds = FERPlusDataset(train_rows, ac_train_tf)
ac_val_ds = FERPlusDataset(val_rows, ac_eval_tf)
ac_test_ds = FERPlusDataset(test_rows, ac_eval_tf)

ac_train_loader = DataLoader(ac_train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
ac_val_loader = DataLoader(ac_val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
ac_test_loader = DataLoader(ac_test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

torch.manual_seed(42)
np.random.seed(42)
cbam_ac = ResNet50_CBAM(num_classes=8).to(device)
print('cbam_ac ready')


### Stage 1 (head only) - M4c

In [ ]:
emotion_cbam_ac = os.path.join(save_dir, 'emotion_cbam_ac.pt')

set_requires_grad(cbam_ac, False)
set_requires_grad(cbam_ac.classifier, True)
opt = torch.optim.Adam([{'params': cbam_ac.classifier.parameters()}], lr=1e-3)

best_f1_ac = 0.0
hist_ac = {'loss': [], 'val_loss': []}

for epoch in range(4):
    tr_loss, tr_acc = train_one_epoch(cbam_ac, ac_train_loader, criterion, opt, device)
    va_loss, va_acc, va_f1, _, _ = evaluate(cbam_ac, ac_val_loader, criterion, device)
    hist_ac['loss'].append(tr_loss)
    hist_ac['val_loss'].append(va_loss)
    print(f's1 e{epoch+1}/4  train {tr_loss:.4f}/{tr_acc:.4f}  val {va_loss:.4f}/{va_acc:.4f} f1 {va_f1:.4f}   [ac]')
    if va_f1 > best_f1_ac:
        best_f1_ac = va_f1
        torch.save(cbam_ac.state_dict(), emotion_cbam_ac)
        print('  saved best')


### Stage 2 (full fine-tune) - M4c

In [ ]:
set_requires_grad(cbam_ac, True)
opt = make_optimizer(cbam_ac, backbone_lr=1e-4, head_lr=1e-3)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=2)

for epoch in range(8):
    tr_loss, tr_acc = train_one_epoch(cbam_ac, ac_train_loader, criterion, opt, device)
    va_loss, va_acc, va_f1, _, _ = evaluate(cbam_ac, ac_val_loader, criterion, device)
    sched.step(va_loss)
    hist_ac['loss'].append(tr_loss)
    hist_ac['val_loss'].append(va_loss)
    print(f's2 e{epoch+1}/8  train {tr_loss:.4f}/{tr_acc:.4f}  val {va_loss:.4f}/{va_acc:.4f} f1 {va_f1:.4f}   [ac]')
    if va_f1 > best_f1_ac:
        best_f1_ac = va_f1
        torch.save(cbam_ac.state_dict(), emotion_cbam_ac)
        print('  saved best')

plt.figure(figsize=(6, 3))
plt.plot(hist_ac['loss'], label='train')
plt.plot(hist_ac['val_loss'], label='val')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.title('ac loss curve')
plt.tight_layout()
plt.show()
print('ac best val macro-F1:', round(best_f1_ac, 4))


### Test eval + snapshot - M4c

In [ ]:
import json as _json

cbam_ac.load_state_dict(torch.load(emotion_cbam_ac, map_location=device))
cbam_ac.eval()
test_loss_ac, test_acc_ac, test_f1_ac, y_true_ac, y_pred_ac = evaluate(cbam_ac, ac_test_loader, criterion, device)

print(f'CBAM aspect test acc {test_acc_ac:.4f}  macro-F1 {test_f1_ac:.4f}')
report_ac = classification_report(y_true_ac, y_pred_ac, labels=list(range(8)), target_names=CLS_NAMES, output_dict=True, zero_division=0)
print(classification_report(y_true_ac, y_pred_ac, labels=list(range(8)), target_names=CLS_NAMES, zero_division=0))

snapshot_ac = {
    'model': 'CBAM aspect',
    'test_accuracy': test_acc_ac,
    'test_macro_f1': test_f1_ac,
    'per_class': {k: v for k, v in report_ac.items() if k in CLS_NAMES},
}
with open(os.path.join(save_dir, 'm4c_aspect_results.json'), 'w') as f:
    _json.dump(snapshot_ac, f, indent=2)
print('saved m4c_aspect_results.json')


## M4 summary &mdash; the preprocessing confounder table

Four snapshots now exist on Drive: baseline (M3), CBAM&times;3 preprocessing modes. Final verdict below: rows = model + preprocessing, columns = test accuracy / macro-F1 / per-class F1. This table mirrors the deepfake experiment design &mdash; the goal is to *measure* preprocessing, not to assume it.


In [ ]:
import json as _json
import pandas as pd

sources = [
    ('baseline (plain 224)', 'm3_baseline_results.json'),
    ('CBAM (plain 224)', 'm4_cbam_results.json'),
    ('CBAM (face crop)', 'm4b_fd_results.json'),
    ('CBAM (aspect/zoom)', 'm4c_aspect_results.json'),
]

rows_acc_f1 = []
per_class = {c: [] for c in CLS_NAMES}
for name, fn in sources:
    d = _json.load(open(os.path.join(save_dir, fn)))
    rows_acc_f1.append((name, d['test_accuracy'], d['test_macro_f1']))
    for c in CLS_NAMES:
        per_class[c].append(d['per_class'][c]['f1-score'])

acc_f1_df = pd.DataFrame(rows_acc_f1, columns=['model', 'test_acc', 'macro_F1'])
print(acc_f1_df.round(4).to_string(index=False))

per_class_df = pd.DataFrame(per_class, index=[n for n, _ in sources])
print('\nper-class F1:')
print(per_class_df.round(3).to_string())

best = rows_acc_f1[0]
for name, acc, f1 in rows_acc_f1[1:]:
    win = 'BOTH beats baseline' if (acc - best[1] > 0 and f1 - best[2] > 0) else ('acc only' if (acc - best[1] > 0) else 'no win')
    print(f'{name:26s} delta acc {(acc - best[1]):+.4f}  delta F1 {(f1 - best[2]):+.4f}  -> {win}')


### M4 verdict

| model | test acc | macro-F1 |
|---|---:|---:|
| baseline (plain 224) | 0.8626 | 0.7269 |
| CBAM (plain 224) | 0.8699 | 0.7115 |
| CBAM (face crop) | 0.8677 | 0.7601 |
| CBAM (aspect/zoom) | 0.8824 | 0.7488 |

1. **CBAM helps when paired with good preprocessing**: aspect/zoom +1.98pt acc / +2.2pt F1; face-crop +0.51pt acc / +3.3pt F1. The plain-CBAM F1 regression (-1.5pt) is one 7-sample disgust noise cell (0.286 vs 0.545); on the six stable classes and by best-val, CBAM is equal or ahead everywhere.
2. **Preprocessing is again the dominant knob**: aspect/zoom moves accuracy (+2.0pt) more than CBAM itself (+0.7pt) - the deepfake M7.5 finding reproduced on a second, unrelated dataset.
3. **App winner**: CBAM + aspect/zoom (best acc, 2nd-best macro-F1, deterministic, no detector dependency). Face-detect stays available as an optional app mode.
4. **Limitations**: disgust (7) / contempt (9) test supports make their per-class F1s +/-noise; stable-6 macro is ~0.83-0.84 across all four rows, so the honest summary is 'CBAM wins both metrics in 2/3 preprocessing regimes and ties on the 6 reliable classes; preprocessing choice matters more.'


In [ ]:
### M5 -- app calibration: temperature scaling + reliability report
# Real-system requirement: confidence must be *calibrated* before we show it next to a label.
# We fit ONE scalar temperature T on val logits (minimizes NLL of softmax(logits/T)),
# rescale by 1/T at inference, and report ECE (15-bin) before/after for both checkpoints.
# Output: save_dir/emotion_temps.json -- copy it next to the app as checkpoints/emotion_temps.json.
import json
from scipy.optimize import minimize_scalar

def _logits(model, loader, device):
    model.eval()
    L, Y = [], []
    with torch.no_grad():
        for x, y in loader:
            L.append(model(x.to(device)).cpu())
            Y.append(y)
    return torch.cat(L), torch.cat(Y)

def _ece(logits, labels, bins=15):
    p = torch.softmax(logits, -1)
    conf, pred = p.max(-1)
    acc = (pred == labels).float()
    edges = torch.linspace(0, 1, bins + 1)
    e, tot = 0.0, 0
    for i in range(bins):
        m = (conf > edges[i]) & (conf <= edges[i + 1])
        if m.sum() == 0:
            continue
        e += m.sum() * abs(conf[m].mean() - acc[m].mean())
        tot += m.sum().item()
    return (e / max(tot, 1)).item()

def _fit_temp(logits, labels, lo=0.4, hi=5.0):
    def nll(t):
        p = torch.softmax(logits / t, -1)
        eps = 1e-12
        return -(torch.gather(p, 1, labels[:, None]) + eps).log().mean().item()
    res = minimize_scalar(nll, bounds=(lo, hi), method='bounded')
    return res.x, res.fun

records = {}
for name, model, loader in [
    ('emotion_cbam_aspect.pt', cbam_ac, ac_val_loader),
    ('emotion_baseline.pt', baseline, val_loader),
]:
    L, Y = _logits(model, loader, device)
    t, _ = _fit_temp(L, Y)
    records[name] = {
        'T': round(float(t), 4),
        'ece_before': round(_ece(L, Y), 4),
        'ece_after': round(_ece(L / t, Y), 4),
        'acc': round(float(((L.max(-1)[1] == Y).float().mean())), 4),
    }
    print(name, records[name])

temps_path = os.path.join(save_dir, 'emotion_temps.json')
json.dump(records, open(temps_path, 'w'), indent=2)
print('saved', temps_path)
from google.colab import files
files.download(temps_path)
# Save it next to the app as checkpoints/emotion_temps.json


### M7 -- real-photo adaptation (RAF-DB)

The FER+ model confidently predicts `neutral` on real photos of smiling people (observed on a test selfie). Root cause: FER+ is 48x48 grayscale aligned faces; modern color photos are a different distribution, so the model is *confidently wrong*, not just overconfident. The real-system fix is to teach the model the target distribution, not to tune the display.

We fine-tune the CBAM aspect/zoom checkpoint on **RAF-DB** (Real-world Affective Faces): ~20k in-the-wild color faces labeled by ~40 annotators each, 7 basic emotions = FER+ **minus contempt**. Label mapping: `anger, disgust, fear, happiness, neutral, sadness, surprise`. Training mixes FER+ (keeps 8-class, avoids forgetting contempt) with RAF-DB (learns real photos). We then re-fit temperature on the real-photo val split.


In [ ]:
### M7 SELF-CONTAINED DRIVER -- cold-runtime safe, no other milestone cell needed
# Builds everything M7 uses from scratch: model classes, train helpers, transforms,
# FER+ rows, RAF-DB loaders, temperature helpers. Requires only Drive mount permission
# and the artifact files (ferplus_rows.pkl, emotion_cbam_ac.pt) already on Drive.

import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
from tqdm.auto import tqdm
from scipy.optimize import minimize_scalar
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from collections import Counter

from google.colab import drive
try:
    from datasets import load_dataset
except ImportError:
    !pip install -q datasets
    from datasets import load_dataset

CLS_NAMES = ['neutral', 'happiness', 'surprise', 'sadness', 'anger', 'disgust', 'fear', 'contempt']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 64
num_workers = 2
print('device:', device)

drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/emotion_detection'
os.makedirs(save_dir, exist_ok=True)
print('drive dir:', save_dir)

# --- model classes (M1) ---
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        hidden = max(in_channels // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, in_channels),
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        b, c, _, _ = x.size()
        avg = self.avg_pool(x).view(b, c)
        max_ = self.max_pool(x).view(b, c)
        attn = self.mlp(avg) + self.mlp(max_)
        attn = self.sigmoid(attn).view(b, c, 1, 1)
        return x * attn

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        max_ = torch.max(x, dim=1, keepdim=True)[0]
        attn = torch.cat([avg, max_], dim=1)
        attn = self.sigmoid(self.conv(attn))
        return x * attn

class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel = ChannelAttention(in_channels, reduction)
        self.spatial = SpatialAttention(kernel_size)
    def forward(self, x):
        x = self.channel(x)
        x = self.spatial(x)
        return x

class ResNet50_CBAM(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.cbam3 = CBAM(1024)
        self.cbam4 = CBAM(2048)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.cbam3(x)
        x = self.layer4(x)
        x = self.cbam4(x)
        x = self.avgpool(x)
        return self.classifier(x)

# --- train helpers (M3) ---
def set_requires_grad(module, value):
    for p in module.parameters():
        p.requires_grad = value

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for images, labels in tqdm(loader, desc='train', leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct += (out.argmax(1) == labels).sum().item()
        n += len(labels)
    return total_loss / n, correct / n

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='eval', leave=False):
            images, labels = images.to(device), labels.to(device)
            out = model(images)
            total_loss += criterion(out, labels).item() * len(labels)
            preds = out.argmax(1).cpu()
            correct += (preds == labels.cpu()).sum().item()
            n += len(labels)
            all_preds.append(preds)
            all_labels.append(labels.cpu())
    y_true = torch.cat(all_labels).numpy()
    y_pred = torch.cat(all_preds).numpy()
    acc = correct / n
    rep = classification_report(y_true, y_pred, labels=list(range(8)), target_names=CLS_NAMES,
                                output_dict=True, zero_division=0)
    macro_f1 = rep['macro avg']['f1-score']
    return total_loss / n, acc, macro_f1, y_true, y_pred

def make_optimizer(model, backbone_lr, head_lr):
    head_ids = set(id(p) for p in model.classifier.parameters())
    base = [p for p in model.parameters() if id(p) not in head_ids and p.requires_grad]
    head = [p for p in model.classifier.parameters()]
    return torch.optim.Adam([
        {'params': base, 'lr': backbone_lr},
        {'params': head, 'lr': head_lr},
    ])

# --- FER+ rows + dataset (M2/M3) ---
df = pd.read_pickle(os.path.join(save_dir, 'ferplus_rows.pkl'))
train_rows = df[df['Usage'] == 'Training']
test_rows = df[df['Usage'] == 'PrivateTest']

class FERPlusDataset(torch.utils.data.Dataset):
    def __init__(self, df_rows, transform):
        self.rows = df_rows[df_rows['valid']].reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        r = self.rows.iloc[idx]
        pix = np.array([int(v) for v in r['pixels'].split()], dtype=np.uint8).reshape(48, 48)
        img = Image.fromarray(pix).convert('RGB')
        return self.transform(img), int(r['cls'])

ac_train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
ac_eval_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

fer_tr_ds = FERPlusDataset(train_rows, ac_train_tf)
fer_te_ds = FERPlusDataset(test_rows, ac_eval_tf)
fer_tr_loader = DataLoader(fer_tr_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
fer_te_loader = DataLoader(fer_te_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

# class-weighted CE, weights only (capped at 8x, mean-renormalized) -- no sampler
counts = train_rows['cls'].value_counts().reindex(range(8)).fillna(0).to_numpy(dtype=np.float64)
raw_weights = counts.sum() / (8 * counts)
weights = np.minimum(raw_weights, 8.0)
weights = weights / weights.mean()
criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32).to(device))
print('class weights (mean 1, capped at 8):', np.round(weights, 2))

# --- RAF-DB loaders (M7, lazy dataset) ---
raf_labels = ['anger', 'disgust', 'fear', 'happiness', 'neutral', 'sadness', 'surprise']

raf = load_dataset('deanngkl/raf-db-7emotions')
ds = raf['train'] if 'train' in raf else raf[list(raf.keys())[0]]
print('RAF samples:', len(ds), '| features:', ds.features)
feat = ds.features['label']
names = feat.names if hasattr(feat, 'names') else raf_labels
raf_to_idx = {i: CLS_NAMES.index(n) for i, n in enumerate(names)}
print('label map ->', raf_to_idx)

raf_y = np.array(ds['label'])
raf_idx = np.arange(len(ds))
tr_idx, rest = train_test_split(raf_idx, test_size=0.20, random_state=42, stratify=raf_y)
va_idx, te_idx = train_test_split(rest, test_size=0.5, random_state=42, stratify=raf_y[rest])

class RealPhotoDataset(torch.utils.data.Dataset):
    def __init__(self, dset, indices, transform, label_map):
        self.dset = dset
        self.indices = [int(i) for i in indices]
        self.transform = transform
        self.label_map = label_map
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        i = self.indices[idx]
        img = self.dset['image'][i].convert('RGB')
        lab = self.label_map[int(self.dset['label'][i])]
        return self.transform(img), lab

raf_tr_ds = RealPhotoDataset(ds, tr_idx, ac_train_tf, raf_to_idx)
raf_va_ds = RealPhotoDataset(ds, va_idx, ac_eval_tf, raf_to_idx)
raf_te_ds = RealPhotoDataset(ds, te_idx, ac_eval_tf, raf_to_idx)
raf_tr_loader = DataLoader(raf_tr_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
raf_va_loader = DataLoader(raf_va_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
raf_te_loader = DataLoader(raf_te_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

mix_tr_ds = torch.utils.data.ConcatDataset([fer_tr_ds, raf_tr_ds])
mix_tr_loader = DataLoader(mix_tr_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)

print('real train/val/test:', len(raf_tr_ds), len(raf_va_ds), len(raf_te_ds))
print('mix train:', len(mix_tr_ds), '| real-class dist:', Counter(raf_y[tr_idx].tolist()))

# --- temperature helpers (M5) ---
def _logits(model, loader, device):
    model.eval()
    L, Y = [], []
    with torch.no_grad():
        for x, y in loader:
            L.append(model(x.to(device)).cpu())
            Y.append(y)
    return torch.cat(L), torch.cat(Y)

def _ece(logits, labels, bins=15):
    p = torch.softmax(logits, -1)
    conf, pred = p.max(-1)
    acc = (pred == labels).float()
    edges = torch.linspace(0, 1, bins + 1)
    e, tot = 0.0, 0
    for i in range(bins):
        m = (conf > edges[i]) & (conf <= edges[i + 1])
        if m.sum() == 0:
            continue
        e += m.sum() * abs(conf[m].mean() - acc[m].mean())
        tot += m.sum().item()
    return (e / max(tot, 1)).item()

def _fit_temp(logits, labels, lo=0.4, hi=5.0):
    def nll(t):
        p = torch.softmax(logits / t, -1)
        eps = 1e-12
        return -(torch.gather(p, 1, labels[:, None]) + eps).log().mean().item()
    res = minimize_scalar(nll, bounds=(lo, hi), method='bounded')
    return res.x, res.fun

print('bootstrap complete -- ready for the M7 train cell')


In [ ]:
m7_path = os.path.join(save_dir, 'emotion_cbam_real.pt')
m7 = ResNet50_CBAM(num_classes=8).to(device)
m7.load_state_dict(torch.load(os.path.join(save_dir, 'emotion_cbam_ac.pt'), map_location=device))

# stage 1: head only on RAF-DB (adapt classifier to real-photo features)
set_requires_grad(m7, False)
set_requires_grad(m7.classifier, True)
opt = torch.optim.Adam([{'params': m7.classifier.parameters()}], lr=1e-3)
for epoch in range(4):
    tr_loss, tr_acc = train_one_epoch(m7, raf_tr_loader, criterion, opt, device)
    va_loss, va_acc, va_f1, _, _ = evaluate(m7, raf_va_loader, criterion, device)
    print(f's1 e{epoch+1}/4  train {tr_loss:.4f}/{tr_acc:.4f}  val {va_loss:.4f}/{va_acc:.4f} f1 {va_f1:.4f}')

# stage 2: full fine-tune on FER+ + RAF-DB mixed
set_requires_grad(m7, True)
opt = make_optimizer(m7, backbone_lr=1e-4, head_lr=1e-3)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)
best_f1 = 0.0
for epoch in range(8):
    tr_loss, tr_acc = train_one_epoch(m7, mix_tr_loader, criterion, opt, device)
    va_loss, va_acc, va_f1, _, _ = evaluate(m7, raf_va_loader, criterion, device)
    sched.step(va_loss)
    print(f's2 e{epoch+1}/8  train {tr_loss:.4f}/{tr_acc:.4f}  val {va_loss:.4f}/{va_acc:.4f} f1 {va_f1:.4f}', flush=True)
    if va_f1 > best_f1:
        best_f1 = va_f1
        torch.save(m7.state_dict(), m7_path)
        print('  saved best')

m7.load_state_dict(torch.load(m7_path, map_location=device))
eva_real = evaluate(m7, raf_te_loader, criterion, device)
eva_fer = evaluate(m7, fer_te_loader, criterion, device)
print('real test  acc %.4f  macro-F1 %.4f' % (eva_real[1], eva_real[2]))
print('FER+ test  acc %.4f  macro-F1 %.4f  (regression check)' % (eva_fer[1], eva_fer[2]))

from sklearn.metrics import classification_report
print(classification_report(eva_real[3], eva_real[4], labels=list(range(8)), target_names=CLS_NAMES, zero_division=0))

json.dump({'real_test': {'acc': round(float(eva_real[1]), 4), 'macro_f1': round(float(eva_real[2]), 4)},
           'ferplus_test_regression': {'acc': round(float(eva_fer[1]), 4), 'macro_f1': round(float(eva_fer[2]), 4)}},
          open(os.path.join(save_dir, 'm7_real_results.json'), 'w'), indent=2)


In [ ]:
# re-calibrate temperature for the real-photo checkpoint (reuses helpers from the M5 cell)
from google.colab import files
L, Y = _logits(m7, raf_va_loader, device)
t, _ = _fit_temp(L, Y)
temps_path2 = os.path.join(save_dir, 'emotion_temps.json')
temps = json.load(open(temps_path2)) if os.path.exists(temps_path2) else {}
temps['emotion_cbam_real.pt'] = {
    'T': round(float(t), 4),
    'ece_before': round(_ece(L, Y), 4),
    'ece_after': round(_ece(L / t, Y), 4),
    'acc': round(float(((L.max(-1)[1] == Y).float().mean())), 4),
}
json.dump(temps, open(temps_path2, 'w'), indent=2)
print(temps)
files.download(temps_path2)
# copy the downloaded emotion_temps.json AND emotion_cbam_real.pt into the app checkpoints/ folder


In [ ]:
### M5-hardening -- feature-space OOD gate (all classes)
# Problem: on closed-mouth anger (and any unusual real photo) the model is CONFIDENTLY WRONG
# (softmax 1.0 neutral on c/d/man-2 even without temperature). Calibration can't fix a wrong
# label -- only flag that the sample is far from the training distribution.
# This cell builds per-class centroids from the penultimate 2048-d features and derives a
# per-class cosine-sim threshold (5th percentile of in-distribution training samples).
# At inference: nearest-centroid similarity below its threshold -> OUT-OF-DISTRIBUTION -> LOW.
# Depends only on the self-contained bootstrap + training (cells 72/73); saves to Drive.
import random
import numpy as np

def _embed_batch(model, x, device):
    with torch.no_grad():
        x = model.stem(x)
        x = model.layer1(x)
        x = model.layer2(x)
        x = model.layer3(x)
        x = model.cbam3(x)
        x = model.layer4(x)
        x = model.cbam4(x)
        x = model.avgpool(x)
        return x.view(x.size(0), -1)

def _collect_embeddings(model, loader, device):
    model.eval()
    feats = {c: [] for c in range(8)}
    for images, labels in tqdm(loader, desc='embed', leave=False):
        e = _embed_batch(model, images.to(device), device).cpu()
        for i, lab in enumerate(labels):
            feats[int(lab)].append(e[i])
    return feats

feats = _collect_embeddings(m7, mix_tr_loader, device)
centroids = torch.stack([torch.mean(torch.stack(feats[c]), 0) if feats[c] else torch.zeros(1) for c in range(8)])
norm_c = centroids / centroids.norm(dim=1, keepdim=True).clamp_min(1e-12)

thr = np.zeros(8)
sims = {}
for c in range(8):
    if not feats[c]:
        continue
    stack = torch.stack(feats[c])
    norm_s = stack / stack.norm(dim=1, keepdim=True).clamp_min(1e-12)
    sims[c] = (norm_s @ norm_c[c]).numpy()
    thr[c] = float(np.percentile(sims[c], 5))

print('per-class counts:', {c: len(feats[c]) for c in range(8)})
print('5th-pct cosine sim (in-dist floor, HIGH==above):', np.round(thr, 4))

# sanity: in-distribution OOD rate on RAF val (should be ~5% per class definition)
m7.eval()
ood_hits = 0
with torch.no_grad():
    for images, labels in tqdm(raf_va_loader, desc='val-ood', leave=False):
        e = _embed_batch(m7, images.to(device), device).cpu()
        norm_e = e / e.norm(dim=1, keepdim=True).clamp_min(1e-12)
        s = norm_e @ norm_c.t()
        best = s.argmax(1)
        best_sim = s.gather(1, best[:, None]).squeeze(1)
        thr_t = torch.tensor(thr)
        ood_hits += (best_sim < thr_t[best]).sum().item()
print('RAF-val OOD rate: %.2f%%  (target ~5%%; high => thresholds too tight)' % (100.0 * ood_hits / len(raf_va_ds)))

torch.save({'centroids': centroids, 'thresholds': torch.tensor(thr),
            'sims': {c: (sims[c].tolist() if len(sims[c]) else []) for c in range(8)}},
           os.path.join(save_dir, 'emotion_centroids_cbam_real.pt'))
print('saved emotion_centroids_cbam_real.pt to', save_dir)


In [ ]:
### M5-hardening II -- geometric second-opinion (MediaPipe landmarks)
# Goal: a label-independent guard. The CNN is confidently WRONG on closed-mouth anger
# (it maps those faces straight onto its own neutral manifold - proof in local notes),
# so classifier-feature OOD can never see them. This cell uses the only signal the CNN
# DOESN'T see: face geometry (mouth aspect ratio, brow gap/height, eye openness, lip slope)
# measured per class on RAF val. We fit a tiny LDA 'second opinion' and MEASURE whether
# anger is even separable from neutral before wiring it into the app (honest gate: if the
# CV results are ~chance, we do NOT ship it - it would only add noise).
!pip install -q mediapipe
import numpy as np
import mediapipe as mp

# detector: legacy 'solutions' API if present, else modern 'tasks' API. Both return
# NormalizedLandmark lists with the same .x/.y/.z interface, so geo_feat is API-agnostic.
try:
    mp.solutions.face_mesh
    LEGACY = True
except AttributeError:
    LEGACY = False

if LEGACY:
    fm = mp.solutions.face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, refine_landmarks=True)
    def detect(pil):
        res = fm.process(np.array(pil))
        return res.multi_face_landmarks[0].landmark if res.multi_face_landmarks else None
else:
    import urllib.request, os as _os
    tmodel = '/content/face_landmarker.task'
    if not _os.path.exists(tmodel):
        urllib.request.urlretrieve(
            'https://storage.googleapis.com/mediapipe-models/face_landmarker/'
            'face_landmarker/float16/1/face_landmarker.task', tmodel)
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision
    lmo = vision.FaceLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=tmodel),
        running_mode=vision.RunningMode.IMAGE, num_faces=1)
    _lm = vision.FaceLandmarker.create_from_options(lmo)
    def detect(pil):
        rim = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.array(pil))
        res = _lm.detect(rim)
        return res.face_landmarks[0] if res.face_landmarks else None

from sklearn.model_selection import StratifiedKFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, confusion_matrix

def geo_feat(lm, W, H):
    P = lambda i: (lm[i].x * W, lm[i].y * H)
    fw = np.linalg.norm(np.array(P(263)) - np.array(P(33))) + 1e-6
    mouth_w = np.linalg.norm(np.array(P(291)) - np.array(P(61)))
    lips_gap = np.linalg.norm(np.array(P(14)) - np.array(P(13)))
    mar = lips_gap / max(mouth_w, 1e-6)
    brow_dist = np.linalg.norm(np.array(P(300)) - np.array(P(70))) / fw
    brow_h = (np.linalg.norm(np.array(P(70)) - np.array(P(159))) +
              np.linalg.norm(np.array(P(300)) - np.array(P(386)))) / (2 * fw)
    ear_l = np.linalg.norm(np.array(P(159)) - np.array(P(145))) / max(np.linalg.norm(np.array(P(133)) - np.array(P(33))), 1)
    ear_r = np.linalg.norm(np.array(P(386)) - np.array(P(374))) / max(np.linalg.norm(np.array(P(362)) - np.array(P(263))), 1)
    eye_open = (ear_l + ear_r) / 2
    c1, c2 = np.array(P(61)), np.array(P(291))
    slope = (c2[1] - c1[1]) / max(c2[0] - c1[0], 1e-6)
    return np.array([mar, brow_dist, brow_h, eye_open, slope])

X, y = [], []
for i in tqdm(va_idx, desc='mesh', leave=False):
    pil = ds['image'][int(i)].convert('RGB')
    W, H = pil.size
    lm = detect(pil)
    if lm is None:
        continue
    X.append(geo_feat(lm, W, H))
    y.append(raf_to_idx[int(ds['label'][int(i)])])
X = np.array(X); y = np.array(y)
names = CLS_NAMES
print('geo samples:', len(X), '| skipped (no face):', len(va_idx) - len(X))
print('per-class counts:', {names[c]: int((y == c).sum()) for c in range(8)})
print('mean MAR by class:', {names[c]: round(float(X[y == c].mean(0)[0]), 4) for c in range(8) if (y == c).sum()})

# MEASURE: does geometry separate classes at all? 5-fold stratified CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe = make_pipeline(StandardScaler(), LDA())
pred = np.zeros_like(y)
for tr, te in skf.split(X, y):
    pipe.fit(X[tr], y[tr])
    pred[te] = pipe.predict(X[te])
print('\n5-fold CV geometric opinion:')
print('  overall acc %.3f  (chance %.3f)' % (accuracy_score(y, pred), (np.bincount(y) / len(y)).max()))
print('  macro-F1 %.3f' % f1_score(y, pred, average='macro', zero_division=0))
print('  per-class F1:', {names[c]: round(float(f1_score((y == c).astype(int), (pred == c).astype(int))), 3) for c in range(8)})

# THE question: is anger vs neutral separable on geometry alone?
m = (y == names.index('anger')) | (y == names.index('neutral'))
if ((y[m] == names.index('anger')).sum() > 0) and ((y[m] == names.index('neutral')).sum() > 0):
    sub_X = X[m]; sub_y = (y[m] == names.index('anger')).astype(int)
    skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    aucs = []
    for tr, te in skf2.split(sub_X, sub_y):
        p2 = make_pipeline(StandardScaler(), LDA())
        p2.fit(sub_X[tr], sub_y[tr])
        aucs.append(roc_auc_score(sub_y[te], p2.decision_function(sub_X[te])))
    print('\nanger-vs-NEUTRAL geometric AUC: %.3f  (0.5 = no signal; 0.9+ = useful)' % np.mean(aucs))

# per-class prior stats for a potential app guard (only saved if AUC is promising)
means = {c: X[y == c].mean(0).tolist() for c in range(8) if (y == c).sum()}
stds = {c: X[y == c].std(0).tolist() for c in range(8) if (y == c).sum()}
geo_prior = {'means': means, 'stds': stds, 'n': {c: int((y == c).sum()) for c in range(8)},
             'cv_acc': float(accuracy_score(y, pred)), 'macro_f1': float(f1_score(y, pred, average='macro', zero_division=0))}
torch.save(geo_prior, os.path.join(save_dir, 'emotion_geo_prior.pt'))
print('\nsaved emotion_geo_prior.pt to', save_dir, '(used only if separation is real)')

# AUC 0.907 anger-vs-neutral => geometry is learnable. Fit the FINAL model on all# data and serialize the fitted LDA (scaler + standardized class means) so the app# can score user photos offline. posterior ~ softmax over ||standardized feat -# class mean||^2 (LDA under shared covariance).
from google.colab import files

pipe_final = make_pipeline(StandardScaler(), LDA())
pipe_final.fit(X, y)
scaler = pipe_final.named_steps['standardscaler']
lda = pipe_final.named_steps['lineardiscriminantanalysis']
class_means_std = lda.means_ if hasattr(lda, 'means_') else np.zeros((1, 5))

m = X[y == names.index('anger')] if (y == names.index('anger')).any() else None
final_dict = {
    'feat_names': ['MAR', 'brow_dist', 'brow_h', 'eye_open', 'slope'],
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_scale': scaler.scale_.tolist(),
    'class_idx': [int(cls) for cls in np.unique(y)],
    'class_labels': [names[int(cls)] for cls in np.unique(y)],
    'class_means_std': class_means_std.tolist(),
    'n_per_class': {names[c]: int((y == c).sum()) for c in range(8)},
    'cv_acc': float(accuracy_score(y, pred)),
    'macro_f1': float(f1_score(y, pred, average='macro', zero_division=0)),
}
geo_path = os.path.join(save_dir, 'emotion_geo_prior.pt')
torch.save(final_dict, geo_path)
print('\nsaved fitted geo prior (scaler+LDA) to', geo_path)
print('class order:', final_dict['class_idx'], final_dict['class_labels'])
files.download(geo_path)
